# Locked smoke40 inference with Qwen3-VL-8B-Instruct

1. Runtime → Change runtime type → GPU
2. Edit only `DEPLOYMENT_ROOT` and `RESULTS_ROOT` below
3. Run all

This notebook does not select candidates, extract frames, edit the prompt, train parameters, or access source videos. Private labels are not read until the formal 40-candidate completion gate passes.

In [ ]:
# [0] User configuration — edit only the two paths.
DEPLOYMENT_ROOT = "/content/drive/MyDrive/mllm_verifier_exploration/deployment"
RESULTS_ROOT = "/content/drive/MyDrive/mllm_smoke40_results"

MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"  # locked; do not change
SEED = 100  # locked; do not change
print("[0] User configuration")
print(f"DEPLOYMENT_ROOT={DEPLOYMENT_ROOT}")
print(f"RESULTS_ROOT={RESULTS_ROOT}")

In [ ]:
# [1] Mount Google Drive
print("[1] Mount Google Drive")
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# [2] Detect GPU
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path(DEPLOYMENT_ROOT) / "colab"))
import colab_run_smoke40_qwen3vl as smoke40
importlib.reload(smoke40)
smoke40.stage(2, "Detect GPU")
GPU = smoke40.detect_gpu()

In [ ]:
# [3] Locate deployment package
smoke40.stage(3, "Locate deployment package")
if MODEL_NAME != smoke40.MODEL_NAME_LOCKED or SEED != smoke40.SEED_LOCKED:
    raise RuntimeError("LOCKED_CONFIGURATION_CHANGED")
PATHS = smoke40.locate_deployment(DEPLOYMENT_ROOT)
RESULTS = Path(RESULTS_ROOT).expanduser().resolve()
RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
# [4] Verify SMOKE40_LOCKFILE
smoke40.stage(4, "Verify SMOKE40_LOCKFILE")
LOCK_STATE = smoke40.verify_lockfile(PATHS)

In [ ]:
# [5] Verify input hashes and label isolation
smoke40.stage(5, "Verify input hashes")
VERIFIED = smoke40.verify_input_hashes(PATHS, LOCK_STATE)
smoke40.verify_label_leakage(PATHS, VERIFIED["manifest"])
IDENTITY = smoke40.build_identity(PATHS, LOCK_STATE)

In [ ]:
# [6] Install inference dependencies (only after both input gates pass)
smoke40.stage(6, "Install inference dependencies")
PROVISIONAL_MODE, _, _ = smoke40.precision_policy(GPU["total_vram_gib"])
smoke40.install_dependencies(quantized=PROVISIONAL_MODE != "bf16")
ENVIRONMENT = smoke40.environment_versions()
smoke40.write_json(RESULTS / "environment.json", ENVIRONMENT)

In [ ]:
# [7] Select deployment precision by the locked hardware-only policy
smoke40.stage(7, "Select deployment precision")
RUNTIME = smoke40.select_precision(GPU, RESULTS)
smoke40.enforce_resume_precision(RESULTS, RUNTIME["precision_mode"])

In [ ]:
# [8] Load the frozen Qwen3-VL-8B-Instruct checkpoint
smoke40.stage(8, "Load Qwen3-VL-8B-Instruct")
MODEL, PROCESSOR, RUNTIME = smoke40.load_model(RUNTIME)
ENVIRONMENT = smoke40.environment_versions()  # refresh after an allowed BF16 → INT4 fallback
smoke40.enforce_resume_precision(RESULTS, RUNTIME["precision_mode"])
smoke40.write_json(RESULTS / "runtime_config.json", RUNTIME)
PROMPT = PATHS["prompt"].read_text(encoding="utf-8")
ROWS = VERIFIED["manifest"]["candidates"]

In [ ]:
# [9] Run 2-candidate technical preflight (no label access)
smoke40.stage(9, "Run 2-candidate technical preflight")
smoke40.run_preflight(ROWS, PROMPT, PATHS, RESULTS, MODEL, PROCESSOR, RUNTIME, IDENTITY)

In [ ]:
# [10] Run formal 40-candidate inference; valid checkpoints resume automatically
smoke40.stage(10, "Run formal 40-candidate inference")
smoke40.run_formal(ROWS, PROMPT, PATHS, RESULTS, MODEL, PROCESSOR, RUNTIME, IDENTITY)

In [ ]:
# [11] Verify formal completion
smoke40.stage(11, "Verify completion")
FORMAL = smoke40.verify_completion(ROWS, RESULTS, RUNTIME, IDENTITY)

In [ ]:
# [12] Run deterministic replay on five seed-fixed candidates
smoke40.stage(12, "Run deterministic replay")
REPLAY = smoke40.run_replay(ROWS, FORMAL, PROMPT, PATHS, RESULTS, MODEL, PROCESSOR, RUNTIME, IDENTITY)

In [ ]:
# [13] Evaluation — this is the first stage allowed to read private labels
smoke40.stage(13, "Evaluation")
EVALUATION = smoke40.run_evaluation(PATHS, RESULTS)

In [ ]:
# [14] Generate final report and immutable result hash manifest
smoke40.stage(14, "Generate final report")
METADATA_PATH = smoke40.save_runtime_metadata(RESULTS, RUNTIME, ENVIRONMENT, IDENTITY)
REPORT_PATH = smoke40.generate_report(RESULTS, RUNTIME, EVALUATION, REPLAY)
RESULT_LOCK_PATH = smoke40.generate_result_lock(
    RESULTS, RUNTIME, IDENTITY, RESULTS / "smoke40_evaluation.json", METADATA_PATH, REPORT_PATH
)
print(f"Report: {REPORT_PATH}")
print(f"Result lock: {RESULT_LOCK_PATH}")
print("SMOKE40_EXPLORATION_COMPLETE")